# Deployment lifecycle

A practical example of managing the lifecycle of an existing MongoDB Atlas
Local deployment: list deployments, retrieve one, stop and restart it, pause
and resume it, and inspect its logs.

Requires a Docker daemon on the machine running this kernel.

In [ ]:
%pip install atlas-local-lib-py

## List the existing deployments

`list()` returns every local Atlas deployment on this machine, whatever its
state, including ones created outside this library.

In [ ]:
from atlas_local import LocalDeployment

for existing in LocalDeployment.list():
    print(f"{existing.name:<30} {existing.state}")

## Get the deployment to work with

`get_or_create` keeps this notebook re-runnable: it returns the deployment if
it is already there, and creates it the first time.

`LocalDeployment.get(NAME)` retrieves the deployment with the specified name and raises `GetDeploymentError` if no matching deployment exists.

In [ ]:
NAME = "lifecycle-demo"

deployment = LocalDeployment.get_or_create(name=NAME)

print(deployment.name, deployment.state)

## Stop and start a deployment

Stopping shuts down the deployment and releases its published host port, while preserving its data. Starting it again brings it back. If no fixed host port was specified when the deployment was created, Docker assigns a new available port.

The `deployment` object is a snapshot of the deployment at the time it was retrieved, so it does not update automatically. Call `get()` again to retrieve its current state.


In [ ]:
deployment.stop()

print("State before refreshing:", deployment.state)

refreshed_deployment = LocalDeployment.get(NAME)
print("State after refreshing:", refreshed_deployment.state)

deployment.start()

refreshed_deployment = LocalDeployment.get(NAME)
print("State after starting:", refreshed_deployment.state)

## Pause and resume it

Pausing freezes the running processes instead of shutting them down, so
resuming is immediate. It is a different axis from stop/start, and the two do
not mix: a paused deployment has to be resumed with `unpause()`, and calling
`start()` on it fails.

In [ ]:
deployment.pause()
print("after pause:  ", LocalDeployment.get(NAME).state)

deployment.unpause()
print("after unpause:", LocalDeployment.get(NAME).state)

Every lifecycle method is also available as a static method taking a name or a
container ID, which is useful when there is no object at hand:

In [ ]:
LocalDeployment.stop_deployment(NAME)
print("after stop_deployment: ", LocalDeployment.get(NAME).state)

LocalDeployment.start_deployment(NAME)
print("after start_deployment:", LocalDeployment.get(NAME).state)

## Inspect the logs

`logs()` returns the container output as a list of lines, most recent last.
Without `tail` it returns everything, which is a lot for a deployment that has
been running for a while.

In [ ]:
for line in deployment.logs(tail=5, timestamps=True):
    print(line.strip())

It can also be narrowed down to a time range, to see only what happened after
the restarts above:

In [ ]:
from datetime import datetime, timedelta, timezone

since = datetime.now(timezone.utc) - timedelta(minutes=1)
recent = deployment.logs(since=since)

print(f"{len(recent)} lines in the last minute")

## Clean up

Deleting removes the container and its data. Skip this cell to keep the
deployment for the next run.

In [ ]:
deployment.delete()

print(NAME in [existing.name for existing in LocalDeployment.list()])